In [ ]:
import pandas as pd
from datetime import datetime
import dukascopy_python as dc
from pathlib import Path

instrument = "EUR/USD"   # change to "GBP/USD" for cable, "USD/JPY", etc.
year = 2025

# Save directly to OneDrive so the data is backed up and usable from other projects
base_dir = Path(r"C:\Users\benc1\OneDrive\claude\Tick Data")
symbol = instrument.replace("/", "")   # "EURUSD" -> used in filenames
out_dir = base_dir / "ticks"
out_dir.mkdir(parents=True, exist_ok=True)

# Download month by month, saving each to disk (resumable)
for month in range(1, 13):
    out = out_dir / f"{symbol}_{year}_{month:02d}.parquet"
    if out.exists():          # skip already-downloaded months
        print(f"Skipping {symbol} {year}-{month:02d} (already downloaded)")
        continue

    start = datetime(year, month, 1)
    # End is the first day of the next month (handles December correctly)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)

    print(f"Downloading {symbol} {start:%Y-%m}...")

    month_df = dc.fetch(
        instrument=instrument,
        interval=dc.INTERVAL_TICK,
        start=start,
        end=end,
        offer_side=dc.OFFER_SIDE_BID,
    )
    month_df.to_parquet(out)

# Combine all months from disk into one DataFrame
parts = sorted(out_dir.glob(f"{symbol}_{year}_*.parquet"))
df = pd.concat(pd.read_parquet(p) for p in parts)
print(f"\nFinished! Total Ticks: {len(df):,}")

# Save the combined result alongside the monthly files
df.to_parquet(base_dir / f"{symbol}_{year}_ticks.parquet")